# StudyChat A100 재처리 — 32K 이하, 배치 자동 비교

세션을 다시 시작하고 위에서부터 실행합니다. 기존 성공과 이전 A100 재처리 성공을 보존합니다.
입력 32,000토큰 초과는 추론하지 않습니다. 전체 대화를 자르거나 요약하지 않습니다.
팀원 방식: 길이 캐시, 길이/대화별 정렬, 동일 표본으로 배치 2·4·8 비교, 표본 성공 재사용.
출력은 2,048토큰을 유지하고 최대 시도는 3회에서 2회로 줄였습니다.
모델은 기존 A100 실행과 같은 공식 BF16 리비전을 재사용합니다.
실제 GPU 속도는 미검증입니다. 메모리/CUDA 오류가 나면 비교를 중단하고 원인 로그를 확인합니다.


## 1. GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경 → GPU로 설정한 뒤 다시 실행하세요.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("VRAM GiB:", round(gpu.total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if gpu.total_memory / 2**30 < 14:
    raise RuntimeError("이 노트북의 9B 시험 설정에는 14 GiB 이상 GPU를 권장합니다. 설정을 먼저 점검하세요.")


## 2. 기존 Drive·ZIP 연결

In [ ]:
# ruff: noqa: F821
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ThinkGuard_NLP")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "qwen35_9b_vllm_fp8_v1"  # 같은 설정으로 재개할 때는 바꾸지 마세요.
BUNDLE_NAME = "thinkguard_colab_vllm_bundle.zip"
BUNDLE = PROJECT_DIR / BUNDLE_NAME
WORK = Path("/content/thinkguard_colab_vllm")
WORK.mkdir(exist_ok=True)
if not BUNDLE.exists():
    print(f"제공받은 {BUNDLE_NAME} 하나를 업로드하세요.")
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise ValueError("파일 이름을 확인하세요.")
    BUNDLE.write_bytes(uploaded[BUNDLE_NAME])
    del uploaded
shutil.copyfile(BUNDLE, WORK / "bundle.zip")
with zipfile.ZipFile(WORK / "bundle.zip") as archive:
    if archive.testzip() is not None:
        raise ValueError("ZIP checksum error")
    for name in archive.namelist():
        if Path(name).name != name or name.startswith("."):
            raise ValueError("Unexpected archive path")
    archive.extractall(WORK)
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
for name, expected in manifest["files"].items():
    with (WORK / name).open("rb") as stream:
        observed = hashlib.file_digest(stream, "sha256").hexdigest()
    if observed != expected:
        raise ValueError(f"업로드 파일 검증 실패: {name}")
sys.path.insert(0, str(WORK))
print("원본 행 수:", manifest["counts"], "진단 표본:", manifest["references"])
print("Drive 저장 위치:", PROJECT_DIR)


## 3. 기존 버전 설치 — 재시작 안내 시 1번부터 다시 실행

In [ ]:
# ruff: noqa: F821
import importlib.metadata
import subprocess
import sys

packages = ["vllm", "torch", "transformers", "safetensors", "sentencepiece", "huggingface-hub"]
environment_file = PROJECT_DIR / "colab_vllm_package_versions.json"


def versions():
    result = {}
    for name in packages:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result


before = versions()
if environment_file.exists():
    previous = json.loads(environment_file.read_text(encoding="utf-8"))
    requirements = [f"{name}=={previous[name]}" for name in packages]
else:
    # Let the vLLM wheel select its matching PyTorch/CUDA dependencies.
    requirements = ["vllm>=0.17.0", "sentencepiece", "huggingface-hub"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *requirements])
installed = {name: importlib.metadata.version(name) for name in packages}
environment_file.write_text(json.dumps(installed, indent=2), encoding="utf-8")
print(installed)
restart_marker = WORK / "restart_required.json"
if before != installed:
    restart_marker.write_text(json.dumps({"pid": __import__("os").getpid()}), encoding="utf-8")
if restart_marker.exists():
    if json.loads(restart_marker.read_text())["pid"] == __import__("os").getpid():
        raise RuntimeError(
            "설치 완료. 런타임 → 세션 다시 시작 후 셀 1부터 다시 실행하세요. 지금 모델을 로드하지 마세요."
        )
    restart_marker.unlink()
print("vLLM 설치 확인 완료. 다음 셀을 실행하세요.")


## 4. 실패 건 선택 — 기존 성공 건은 제외

In [ ]:
"""Validated result reuse and length-aware scheduling for the 32K retry pass."""

from journal import record_key, stable_hash


def validate_attempt(records, config, original):
    checked = {}
    values = records.values() if isinstance(records, dict) else records
    for record in values:
        key = record_key(record)
        if key in checked or key not in original:
            raise ValueError("Duplicate/unexpected retry ID: " + key)
        source = original[key]
        if source["dataset"] != "studychat" or source["status"] == "ok":
            raise ValueError("Retry includes non-target/original success: " + key)
        if record.get("run_signature") != stable_hash(config):
            raise ValueError("Retry config signature mismatch: " + key)
        if any(
            record.get(k) != source.get(k) for k in ("input_sha256", "source_sha256", "student_id", "conversation_id")
        ):
            raise ValueError("Retry source identity mismatch: " + key)
        checked[key] = record
    return checked


def reuse_successes(attempts, original):
    successes, configs = {}, {}
    for records, config in attempts:
        signature = stable_hash(config)
        configs[signature] = config
        for key, record in validate_attempt(records, config, original).items():
            if record["status"] == "ok" and key not in successes:
                successes[key] = record
    return successes, configs


def schedule_32k(sources, counts, cutoff=32000):
    if len(sources) != len(counts) or any(type(n) is not int or n <= 0 for n in counts):
        raise ValueError("Invalid token length cache")
    lengths, eligible, excluded = {}, [], []
    for source, count in zip(sources, counts, strict=False):
        key = record_key(source)
        if key in lengths:
            raise ValueError("Duplicate source ID")
        lengths[key] = count
        (eligible if count <= cutoff else excluded).append(source)
    eligible.sort(
        key=lambda s: (lengths[record_key(s)] // 2048, s["conversation_id"], lengths[record_key(s)], record_key(s))
    )
    return eligible, excluded, lengths


def representative_sample(sources, lengths, count=8):
    ordered = sorted(sources, key=lambda s: (lengths[record_key(s)], record_key(s)))
    count = min(count, len(ordered))
    if count <= 1:
        return ordered[:count]
    return [ordered[(len(ordered) - 1) * i // (count - 1)] for i in range(count)]

"""Select failed StudyChat rows and preserve original successes during retry merge."""

from collections import Counter

from journal import record_key, stable_hash


def select_retry_sources(inputs, original, max_tokens):
    selected, seen = [], set()
    for source in inputs:
        key = record_key(source)
        previous = original.get(key)
        if previous is None or source["dataset"] != "studychat" or previous["status"] == "ok":
            continue
        if key in seen:
            raise ValueError("Duplicate retry source")
        seen.add(key)
        if any(previous.get(k) != source[k] for k in ("input_sha256", "source_sha256")):
            raise ValueError("Original result/input hash mismatch: " + key)
        if previous["status"] not in {"context_overflow", "invalid_output", "oom"}:
            continue
        selected.append(source)
    # Original counts are estimates for ordering only. Tokenizer checks the actual
    # current prompt (including repairs) before every engine call.
    selected.sort(key=lambda s: (original[record_key(s)].get("input_tokens", 0), record_key(s)))
    deferred = [record_key(s) for s in selected if original[record_key(s)].get("input_tokens", 0) > max_tokens]
    return selected, deferred


def merge_retry_results(original, retries, retry_config):
    merged = dict(original)
    recovered = 0
    for key, result in retries.items():
        previous = original.get(key)
        if key != record_key(result) or previous is None or previous["dataset"] != "studychat":
            raise ValueError("Unexpected retry result: " + key)
        if previous["status"] == "ok":
            raise ValueError("Retry attempted to replace original success: " + key)
        if result.get("run_signature") != stable_hash(retry_config):
            raise ValueError("Retry signature mismatch")
        if any(previous.get(k) != result.get(k) for k in ("input_sha256", "source_sha256")):
            raise ValueError("Retry input mismatch")
        if result["status"] == "ok":
            merged[key] = result
            recovered += 1
    report = {
        "processed": len(merged),
        "original_ok_preserved": sum(r["status"] == "ok" for r in original.values()),
        "recovered": recovered,
        "statuses": dict(Counter(r["status"] for r in merged.values())),
        "remaining_failed_ids": [k for k, r in merged.items() if r["status"] != "ok"],
        "note": "Mixed inference profiles; consult provenance.json. ok is validation, not human gold.",
    }
    return merged, report

import json
from collections import Counter
from journal import Journal, atomic_json, record_key, stable_hash
from runner import read_inputs, run, configuration, export_review

BASE_ROOT = PROJECT_DIR / "qwen35_9b_vllm_fp8_v1"
PREVIOUS_ROOT = PROJECT_DIR / "qwen35_9b_a100_bf16_retry_v1"
RETRY_ROOT = PROJECT_DIR / "qwen35_9b_a100_32k_optimized_v1"
RETRY_ROOT.mkdir(exist_ok=True)
base_config = json.loads((BASE_ROOT / "full/run_config.json").read_text(encoding="utf-8"))
if base_config["bundle_inputs_sha256"] != manifest["inputs_sha256"]:
    raise ValueError("Original result and input ZIP differ")
original_journal = Journal(BASE_ROOT / "full", base_config)
original = original_journal.records
MAX_INPUT = 32000
MAX_OUTPUT = 2048
BATCH_SIZE = 8  # 변경 시 RETRY_ROOT도 새 이름으로 변경하세요.
retry_sources, estimated_deferred = select_retry_sources(
    read_inputs(WORK / "inputs.jsonl.gz"), original, MAX_INPUT)
if not retry_sources:
    raise RuntimeError("재처리할 StudyChat 실패 건이 없습니다.")
print("기존 결과:", original_journal.summary())
print("실패 건만 선택:", len(retry_sources), "/ 기존 길이 기준 한도 초과:", len(estimated_deferred))
atomic_json(RETRY_ROOT / "selection.json", {"original_signature": stable_hash(base_config),
            "retry_ids": [record_key(s) for s in retry_sources], "estimated_deferred_ids": estimated_deferred})

attempts = []
config_registry = {}
for directory in (PREVIOUS_ROOT / "full", PREVIOUS_ROOT / "retry_32k_only_v1/full"):
    config_path = directory / "run_config.json"
    if config_path.exists():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if any(config.get(k) != base_config.get(k) for k in (
            "bundle_inputs_sha256", "source_hashes", "prompt_version", "prompt_sha256")):
            raise ValueError("Previous attempt uses different data/prompt")
        saved = Journal(directory, config)
        attempts.append((saved.records, config))
reused, config_registry = reuse_successes(attempts, original)
retry_sources = [s for s in retry_sources if record_key(s) not in reused]
print("이전 재처리 성공 재사용:", len(reused), "/ 남은 실패:", len(retry_sources))
# Reuse the exact pinned BF16 revision, including across future reconnections.
pin_path = RETRY_ROOT / "model_revision.json"
previous_pin = PREVIOUS_ROOT / "model_revision.json"
if previous_pin.exists() and not pin_path.exists():
    atomic_json(pin_path, json.loads(previous_pin.read_text(encoding="utf-8")))
if attempts and not pin_path.exists():
    raise ValueError("Previous model pin is missing; do not silently choose a new revision")
length_identity = stable_hash({"inputs": manifest["inputs_sha256"],
    "keys": [record_key(s) for s in retry_sources], "prompt": base_config["prompt_sha256"],
    "model_pin": json.loads(pin_path.read_text(encoding="utf-8")) if pin_path.exists() else None,
    "transformers": installed["transformers"]})


## 5. A100 모델 로드 — 기존 모델 로드 셀 대신 이것만 실행

In [ ]:
import os
import time
import importlib.metadata
from functools import partial
from huggingface_hub import HfApi
import torch
from contract import chat_messages, parse_annotation

if "model" in globals():
    raise RuntimeError("이미 모델이 있습니다. 세션을 다시 시작하고 1번부터 실행하세요.")
if (WORK / "restart_required.json").exists():
    raise RuntimeError("설치 후 세션을 다시 시작하고 1번부터 실행하세요.")
if "A100" not in torch.cuda.get_device_name(0) or torch.cuda.get_device_properties(0).total_memory / 2**30 < 70:
    raise RuntimeError("이 설정은 A100 80GB용입니다. 할당된 GPU를 확인하세요.")
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
from vllm import LLM
MODEL_ID = "Qwen/Qwen3.5-9B"
pin_path = RETRY_ROOT / "model_revision.json"
if pin_path.exists():
    model_pin = json.loads(pin_path.read_text(encoding="utf-8"))
else:
    model_pin = {"model_id": MODEL_ID, "revision": HfApi().model_info(MODEL_ID).sha}
    atomic_json(pin_path, model_pin)
if model_pin["model_id"] != MODEL_ID:
    raise ValueError("Unexpected model pin")
settings = dict(dtype="bfloat16", max_model_len=MAX_INPUT + MAX_OUTPUT,
                max_num_seqs=BATCH_SIZE, max_num_batched_tokens=16384,
                gpu_memory_utilization=0.90, language_model_only=True,
                enable_chunked_prefill=True, enable_prefix_caching=True, enforce_eager=True, seed=0)
model = LLM(model=MODEL_ID, revision=model_pin["revision"], tokenizer_revision=model_pin["revision"],
            trust_remote_code=False, **settings)
tokenizer = model.get_tokenizer()
metadata = {"model_id": MODEL_ID, "revision": model_pin["revision"], "engine": "vllm",
            "quantization": "none; BF16", "engine_settings": settings,
            "versions": {p: importlib.metadata.version(p) for p in ("torch", "vllm", "transformers", "huggingface-hub")},
            "gpu": torch.cuda.get_device_name(0), "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30}
print("로드 완료:", metadata)

class Annotator:
    batch_size = 8

    def __init__(self, model, tokenizer, max_input_tokens=6000, max_new_tokens=768):
        self.model, self.tokenizer = model, tokenizer
        self.max_input_tokens, self.max_new_tokens = max_input_tokens, max_new_tokens

    def __call__(self, source):
        return self.annotate_batch([source])[0]

    def annotate_batch(self, sources):
        from vllm import SamplingParams

        started = time.monotonic()
        results = [None] * len(sources)
        messages = [chat_messages(source) for source in sources]
        failures = [[] for _ in sources]
        active = list(range(len(sources)))
        for attempt in range(2):
            prompts, indices = [], []
            for i in active:
                if not sources[i].get("input_verified", True):
                    results[i] = {"status": "input_unverified", "error": "History structure did not validate"}
                    continue
                tokens = self.tokenizer.apply_chat_template(
                    messages[i], add_generation_prompt=True, tokenize=True, return_dict=False, enable_thinking=False
                )
                if not isinstance(tokens, list) or not tokens or any(type(t) is not int or t < 0 for t in tokens):
                    raise TypeError("Chat template must return a nonempty list of nonnegative integer token IDs")
                if len(tokens) > self.max_input_tokens:
                    results[i] = {
                        "status": "context_overflow",
                        "input_tokens": len(tokens),
                        "error": "Full input exceeds budget; no silent truncation",
                    }
                    continue
                prompts.append({"prompt_token_ids": tokens})
                indices.append(i)
            if not prompts:
                break
            # Engine failures abort the run; completed journal checkpoints remain resumable.
            outputs = self.model.generate(
                prompts, SamplingParams(temperature=0, max_tokens=self.max_new_tokens, seed=0), use_tqdm=False
            )
            if len(outputs) != len(indices):
                raise RuntimeError("vLLM returned an unexpected batch size")
            active = []
            for i, request, prompt in zip(indices, outputs, prompts, strict=False):
                completion = request.outputs[0]
                text = completion.text
                try:
                    if completion.finish_reason != "stop":
                        raise ValueError("Generation did not finish normally: " + str(completion.finish_reason))
                    parsed = parse_annotation(text, sources[i])
                    results[i] = {
                        **parsed,
                        "status": "ok",
                        "input_tokens": len(prompt["prompt_token_ids"]),
                        "output_tokens": len(completion.token_ids),
                        "attempts": attempt + 1,
                        "raw_response": text,
                        "validation_failures": list(failures[i]),
                    }
                except (ValueError, TypeError, KeyError, AttributeError) as exc:
                    failures[i].append(str(exc))
                    if attempt < 1:
                        messages[i] += [
                            {"role": "assistant", "content": text},
                            {
                                "role": "user",
                                "content": "Your annotation failed validation: "
                                + str(exc)
                                + ". Return complete JSON including all reasons and an evidence array (empty if appropriate). "
                            + "Copy short evidence quotes verbatim, including punctuation and whitespace, "
                            + "from a STUDENT turn only. Never paraphrase quotes or cite assistant turns. "
                            + "For current attempt=true cite turn=-1 and set cumulative=true. "
                            + "For past-only evidence establish the same task; otherwise use null if uncertain. "
                            + "Do not change judgments merely to pass validation. Keep reasons brief.",
                            },
                        ]
                        active.append(i)
                    else:
                        results[i] = {"status": "invalid_output", "error": "; ".join(failures[i])}
        elapsed = time.monotonic() - started
        for result in results:
            if result is None:
                raise RuntimeError("Missing batch result")
            result["batch_seconds"] = elapsed
            result["batch_size"] = len(sources)
        return results

annotator = Annotator(model, tokenizer, max_input_tokens=MAX_INPUT, max_new_tokens=MAX_OUTPUT)
annotator.batch_size = BATCH_SIZE
retry_config = configuration(manifest, metadata, annotator)
if any(retry_config[k] != base_config[k] for k in ("prompt_version", "prompt_sha256", "source_hashes")):
    raise ValueError("Original prompt/rules/source changed")
retry_config.update(retry_of=stable_hash(base_config), repair_version="exact-evidence-v2-two-attempts",
                    selection_sha256=stable_hash([record_key(s) for s in retry_sources]))
atomic_json(RETRY_ROOT / "current_hardware.json", metadata)
print("배치:", BATCH_SIZE, "입력:", MAX_INPUT, "출력:", MAX_OUTPUT)

for _, config in attempts:
    # Cross-profile reuse is allowed only for the same BF16 weights, prompt and versions.
    for key in ("model_id", "revision", "quantization", "versions"):
        if config["model"].get(key) != metadata.get(key):
            raise ValueError("Previous successful attempt uses a different model/environment: " + key)


## 6. 길이 캐시·32K 초과 제외·배치 2/4/8 비교

In [ ]:
from collections import Counter
import time

# Compute actual current prompt lengths once, then persist an identity-checked cache.
length_cache = RETRY_ROOT / "token_lengths.json"
cached = json.loads(length_cache.read_text(encoding="utf-8")) if length_cache.exists() else {}
if cached.get("identity") == length_identity:
    counts = cached["counts"]
    print("저장된 길이 캐시 재사용")
else:
    counts = []
    for index, source in enumerate(retry_sources, 1):
        ids = tokenizer.apply_chat_template(chat_messages(source), add_generation_prompt=True,
            tokenize=True, return_dict=False, enable_thinking=False)
        if not isinstance(ids, list) or not ids or any(type(t) is not int for t in ids):
            raise TypeError("Expected integer token IDs")
        counts.append(len(ids))
        if index % 200 == 0:
            print("입력 길이 계산:", index, "/", len(retry_sources), flush=True)
    atomic_json(length_cache, {"identity": length_identity, "counts": counts})
eligible, excluded, lengths = schedule_32k(retry_sources, counts)
atomic_json(RETRY_ROOT / "excluded_over_32k.json", [
    {"id": record_key(s), "input_tokens": lengths[record_key(s)], "status": "context_overflow"}
    for s in excluded])
print("32K 이하:", len(eligible), "/ 초과 제외:", len(excluded))

def config_for(batch):
    item = Annotator(model, tokenizer, max_input_tokens=32000, max_new_tokens=MAX_OUTPUT)
    item.batch_size = batch
    config = configuration(manifest, metadata, item)
    config.update(retry_of=stable_hash(base_config),
        repair_version="exact-evidence-v2-two-attempts", execution_profile="32k-optimized-v1",
        selection_sha256=stable_hash([record_key(s) for s in eligible]))
    return item, config

# Prefer prior context-overflow cases; known repeated output failures are processed later.
overflow_candidates = [s for s in eligible if original[record_key(s)]["status"] == "context_overflow"]
probe = representative_sample(overflow_candidates or eligible, lengths, 8)
probe_identity = stable_hash({"keys": [record_key(s) for s in probe],
    "configs": {str(b): config_for(b)[1] for b in (2, 4, 8)}})
choice_path = RETRY_ROOT / "batch_selection.json"
choice = json.loads(choice_path.read_text(encoding="utf-8")) if choice_path.exists() else None
measurements, pilots = [], []
for batch in (2, 4, 8):
    item, config = config_for(batch)
    path = RETRY_ROOT / f"benchmark/b{batch}"
    cached_pilot = Journal(path, config)
    if set(cached_pilot.records) != {record_key(s) for s in probe}:
        if not model.reset_prefix_cache():
            raise RuntimeError("Could not clear prefix cache before benchmark")
        pilot = run(probe, item, path, config)
    else:
        pilot = cached_pilot
    checked = validate_attempt(pilot.records, config, original)
    if set(checked) != {record_key(s) for s in probe}:
        raise ValueError("Benchmark has missing/extra IDs")
    seconds = sum(r["batch_seconds"] / r["batch_size"] for r in checked.values())
    ok = sum(r["status"] == "ok" for r in checked.values())
    measurements.append({"batch": batch, "seconds": seconds, "count": len(probe), "ok": ok})
    pilots.append((checked, config))
    print("배치 비교:", measurements[-1], flush=True)
if choice is not None:
    if choice["identity"] != probe_identity:
        raise ValueError("Benchmark identity changed; use a new run name")
    selected_batch = choice["selected_batch"]
    if selected_batch not in (2, 4, 8):
        raise ValueError("Invalid saved batch")
else:
    # Prefer useful successful throughput. Failures are still saved and reported.
    candidates = [m for m in measurements if m["ok"] > 0]
    if probe and not candidates:
        raise RuntimeError("No benchmark candidate produced a valid result; inspect benchmark logs")
    selected_batch = min(candidates, key=lambda m: m["seconds"] / m["ok"])["batch"] if candidates else 2
    atomic_json(choice_path, {"identity": probe_identity, "selected_batch": selected_batch,
        "measurements": measurements, "comparison": "same 8 inputs; reusable successful pilot outputs"})
pilot_successes, pilot_configs = reuse_successes(pilots, original)
reused.update(pilot_successes)
config_registry.update(pilot_configs)
config_registry.update({stable_hash(c): c for _, c in attempts})
selected_annotator, selected_config = config_for(selected_batch)
pending = [s for s in eligible if record_key(s) not in reused]
pending.sort(key=lambda s: (original[record_key(s)]["status"] != "context_overflow",
    lengths[record_key(s)] // 2048, s["conversation_id"], lengths[record_key(s)], record_key(s)))
print("선택 배치:", selected_batch, "/ 성공 재사용:", len(reused), "/ 본 실행:", len(pending))


## 7. 선택 배치로 실행·재개

In [ ]:
# 길이 초과는 제외하고, 과거 context_overflow 건부터 처리합니다.
optimized_journal = run(pending, selected_annotator, RETRY_ROOT / f"full_b{selected_batch}", selected_config)


## 8. 성공 보존·누락/중복/실패 검사 및 병합

In [ ]:
full = Journal(RETRY_ROOT / f"full_b{selected_batch}", selected_config)
checked = validate_attempt(full.records, selected_config, original)
if set(checked) != {record_key(s) for s in pending}:
    raise ValueError("본 실행 결과 누락/추가 ID가 있습니다. 7번을 재개하세요.")
successes, more_configs = reuse_successes([(checked, selected_config)], original)
reused.update(successes)
config_registry.update(more_configs)
merged = dict(original)
merged.update(reused)
if len(merged) != len(original) or set(merged) != set(original):
    raise ValueError("최종 결과에 누락/추가 ID가 있습니다.")
if any(merged[k] != r for k, r in original.items() if r["status"] == "ok"):
    raise ValueError("기존 성공 결과가 변경되었습니다.")
report = {"expected_owner_rows": len(original), "processed": len(merged),
    "original_ok_preserved": sum(r["status"] == "ok" for r in original.values()),
    "retry_successes_reused_or_recovered": len(reused), "excluded_over_32k": len(excluded),
    "statuses": dict(Counter(r["status"] for r in merged.values())),
    "missing_ids": sorted(set(original) - set(merged)), "duplicate_ids": [],
    "all_rows_ok": all(r["status"] == "ok" for r in merged.values()),
    "note": "Owner range only; teammate not included. Existing failed outputs remain if not recovered."}
out = RETRY_ROOT / "merged"
out.mkdir(exist_ok=True)
target = out / "annotations.jsonl"
temp = target.with_suffix(".tmp")
with temp.open("w", encoding="utf-8") as stream:
    for key in sorted(merged):
        stream.write(json.dumps(merged[key], ensure_ascii=False) + "\n")
temp.replace(target)
atomic_json(out / "coverage.json", report)
atomic_json(out / "remaining_errors.json", [r for r in merged.values() if r["status"] != "ok"])
atomic_json(out / "provenance.json", {"original_config": base_config, "retry_configs": config_registry})
print(report)
print("병합 결과:", target)
